# Lab 10: Data File Analyzer (DS-STAR Component)

**Navigation** : [Lab 9 <<](../Day4-Foundations/Lab9-First-ADK-Agent.ipynb) | [Index](../../README.md) | [>> Lab 11](Lab11-Planner-Coder-Loop.ipynb)

## Objectifs d'apprentissage

À la fin de ce laboratoire, vous saurez :
1. Implémenter le module File Analyzer de DS-STAR
2. Analyser des fichiers hétérogènes (CSV, JSON, Markdown, texte)
3. Extraire des métadonnées structurées automatiquement
4. Générer des résumés intelligents avec le LLM

### Prérequis
- Python 3.10+
- Configuration multi-provider active
- Connaissance de Pandas (Lab 4)

### Durée estimée : 30-40 minutes

## 1. Configuration

In [1]:
import sys
from pathlib import Path

# Add parent directory (AgenticDataScience) to path for config/utils imports
sys.path.insert(0, str(Path().resolve().parent))

import os
import json
import pandas as pd
import numpy as np
from dataclasses import dataclass, asdict

from config import get_settings
from utils import LLMClient

Chargement des parametres de configuration.

In [2]:
settings = get_settings()
print(f'Provider: {settings.active_provider}')

Provider: openrouter


## 2. FileMetadata DataClass

In [3]:
@dataclass
class FileMetadata:
    filename: str
    format: str
    size_bytes: int
    num_rows: int = None
    num_columns: int = None
    columns: list = None
    statistics: dict = None
    sample_data: list = None

## 3. FileAnalyzer Class

In [4]:
class FileAnalyzer:
    SUPPORTED = {'.csv': 'csv', '.json': 'json', '.md': 'markdown', '.txt': 'text'}

    def __init__(self, llm_client=None):
        self.llm = llm_client or LLMClient()

    def detect_format(self, path):
        ext = Path(path).suffix.lower()
        return self.SUPPORTED.get(ext, 'unknown')

    def analyze_csv(self, path):
        df = pd.read_csv(path)
        cols = []
        for c in df.columns:
            info = {'name': c, 'dtype': str(df[c].dtype), 'missing_pct': round(df[c].isna().mean()*100, 2)}
            if df[c].dtype in ['int64', 'float64']:
                info['stats'] = {'min': float(df[c].min()), 'max': float(df[c].max()), 'mean': float(df[c].mean())}
            cols.append(info)
        return FileMetadata(
            filename=Path(path).name, format='csv', size_bytes=os.path.getsize(path),
            num_rows=len(df), num_columns=len(df.columns), columns=cols,
            sample_data=df.head(5).to_dict('records')
        )

    def analyze(self, path):
        fmt = self.detect_format(path)
        if fmt == 'csv':
            return self.analyze_csv(path)
        raise ValueError(f"Format non supporte: {fmt}")

    def generate_summary(self, meta):
        lines = [f"Fichier: {meta.filename}", f"Format: {meta.format}", f"Taille: {meta.size_bytes/1024:.1f} KB"]
        if meta.num_rows:
            lines.append(f"Lignes: {meta.num_rows}, Colonnes: {meta.num_columns}")
        if meta.columns:
            for c in meta.columns[:5]:
                lines.append(f"  - {c['name']} ({c['dtype']}): {c.get('missing_pct', 0)}% manquant")
        return "\n".join(lines)

    def generate_llm_summary(self, meta, question=None):
        ctx = self.generate_summary(meta)
        if meta.sample_data:
            ctx += f"\n\nEchantillon:\n{json.dumps(meta.sample_data[:2], indent=2, default=str)[:400]}"
        prompt = f"Analyse ce fichier et resume-le:\n{ctx}\n{f'Question: {question}' if question else ''}"
        return self.llm.generate(prompt, temperature=0.3)

## 4. Tests

In [5]:
# Creation fichier test
import tempfile
test_dir = tempfile.mkdtemp()

df = pd.DataFrame({
    'id': range(1, 101),
    'product': [f'P{i}' for i in range(1, 101)],
    'category': np.random.choice(['A', 'B', 'C'], 100),
    'price': np.random.uniform(10, 500, 100).round(2),
    'qty': np.random.randint(1, 50, 100)
})
df.loc[10:15, 'price'] = np.nan

csv_path = os.path.join(test_dir, 'products.csv')
df.to_csv(csv_path, index=False)
print(f'CSV cree: {csv_path}')

CSV cree: C:\Users\jsboi\AppData\Local\Temp\tmplvqxn4p2\products.csv


Test de l'analyseur de fichiers.

In [6]:
# Test analyseur
analyzer = FileAnalyzer()
meta = analyzer.analyze(csv_path)
print(analyzer.generate_summary(meta))

Fichier: products.csv
Format: csv
Taille: 1.9 KB
Lignes: 100, Colonnes: 5
  - id (int64): 0.0% manquant
  - product (str): 0.0% manquant
  - category (str): 0.0% manquant
  - price (float64): 6.0% manquant
  - qty (int64): 0.0% manquant


Detail des colonnes detectees.

In [7]:
# Colonnes detail
for c in meta.columns:
    print(f"{c['name']}: {c['dtype']}, {c.get('missing_pct', 0)}% manquant")
    if 'stats' in c:
        print(f"  -> min={c['stats']['min']:.1f}, max={c['stats']['max']:.1f}, mean={c['stats']['mean']:.1f}")

id: int64, 0.0% manquant
  -> min=1.0, max=100.0, mean=50.5
product: str, 0.0% manquant
category: str, 0.0% manquant
price: float64, 6.0% manquant
  -> min=17.9, max=498.9, mean=284.2
qty: int64, 0.0% manquant
  -> min=1.0, max=49.0, mean=25.2


## 5. Resume LLM

In [8]:
# Resume intelligent
summary = analyzer.generate_llm_summary(meta)
print(summary)

Voici l'analyse et le résumé de votre fichier **products.csv** :

### 1. Vue d'ensemble
Il s'agit d'un petit jeu de données (1.9 KB) représentant un **catalogue ou un inventaire de produits**. Le fichier est très compact, contenant exactement **100 lignes** (produits) réparties sur **5 colonnes**.

### 2. Description des variables (Colonnes)
D'après l'échantillon et les métadonnées, voici la structure des données :
*   **`id`** *(entier)* : L'identifiant unique de chaque produit (ex: 1, 2).
*   **`product`** *(texte)* : Le nom ou la référence du produit (ex: "P1", "P2").
*   **`category`** *(texte)* : La famille ou catégorie à laquelle appartient le produit (ex: "C").
*   **`price`** *(décimal)* : Le prix unitaire du produit (ex: 488.11, 139.64).
*   **`qty`** *(entier)* : La quantité en stock ou le volume de ventes du produit (ex: 15, 7).

### 3. Qualité des données
Le jeu de données est globalement très propre, mais présente un point d'attention :
*   **Excellente complétude globale 

Generation d'un resume intelligent guide par le LLM.

In [9]:
# Resume guide
q = "Quelles analyses puis-je faire sur ces donnees?"
print(f"Question: {q}\n")
result = analyzer.generate_llm_summary(meta, q)
print(result)

Question: Quelles analyses puis-je faire sur ces donnees?



Voici un résumé et une analyse de votre fichier, ainsi que des propositions d'analyses que vous pouvez réaliser.

### 📝 Résumé du fichier `products.csv`
Il s'agit d'une petite base de données (100 lignes) représentant un **catalogue de produits** ou un **inventaire de stock**. 
Chaque ligne représente un produit unique avec les informations suivantes :
*   **id** : Un identifiant unique pour chaque produit.
*   **product** : Le nom ou la référence du produit (ex: "P1").
*   **category** : La catégorie à laquelle appartient le produit (ex: "C").
*   **price** : Le prix unitaire du produit. **Attention : 6% des produits n'ont pas de prix (valeurs manquantes).**
*   **qty** : La quantité disponible (en stock) ou vendue.

---

### 💡 Quelles analyses pouvez-vous faire sur ces données ?

Voici plusieurs axes d'analyse, du plus simple au plus avancé, que vous pouvez explorer avec ce jeu de données :

#### 1. Nettoyage et préparation des données (Priorité)
Avant toute analyse financière, il fa

## 6. Resume du Lab### Points cles1. FileAnalyzer: Analyse automatique de fichiers2. Metadonnees: Extraction de schemas et stats3. Resume LLM: Contextualisation pour le Planner### Prochaine etape- Lab 11: Boucle Planner-Coder-Verifier

In [10]:
# Cleanup
import shutil
shutil.rmtree(test_dir)
print('Done')

Done


## Exercice

À vous d'étendre le FileAnalyzer pour gérer un nouveau format de fichier !


In [11]:
# Exercice : Etendez le FileAnalyzer pour gerer les fichiers JSON
# 1. Ajoutez une methode analyze_json
# 2. Testez-la sur un fichier JSON de test

import json
import tempfile

# Creation d'un fichier JSON de test
exercise_dir = tempfile.mkdtemp()
test_json_path = os.path.join(exercise_dir, 'test_data.json')
test_data = {
    'products': [
        {'id': 1, 'name': 'Laptop', 'price': 999.99, 'stock': 15},
        {'id': 2, 'name': 'Mouse', 'price': 19.99, 'stock': 50},
        {'id': 3, 'name': 'Keyboard', 'price': 49.99, 'stock': 30}
    ],
    'metadata': {
        'source': 'inventory_system',
        'last_updated': '2026-03-13'
    }
}

with open(test_json_path, 'w') as f:
    json.dump(test_data, f)

# Exercice: Implementez la classe ExtendedFileAnalyzer
# Elle doit heriter de FileAnalyzer et ajouter une methode analyze_json
class ExtendedFileAnalyzer(FileAnalyzer):
    def analyze_json(self, path):
        """Analyse un fichier JSON et extrait les metadonnees."""
        # Exercice: Ouvrez et chargez le fichier JSON
        # Indice: with open(path, 'r') as f: data = json.load(f)
        data = None  # Remplacez None

        # Exercice: Comptez les cles au niveau racine
        root_keys = None  # Indice: list(data.keys())

        # Exercice: Trouvez la plus grande liste dans les valeurs
        # Indice: parcourez data.items(), testez isinstance(value, list)
        max_list = 0
        list_name = None
        # ... votre code ici

        # Exercice: Retournez un FileMetadata avec les informations extraites
        return None  # Remplacez par FileMetadata(...)

# Exercice: Testez l'analyseur etendu
# extended_analyzer = ExtendedFileAnalyzer()
# json_meta = extended_analyzer.analyze_json(test_json_path)
# print(f"Fichier: {json_meta.filename}")
# print(f"Cles racine: {json_meta.columns}")

# Nettoyage
# os.remove(test_json_path)

print("Exercice a completer")

Exercice a completer


## Conclusion

Ce notebook a permis d'explorer les aspects essentiels de lab10 file analyzer. Les points cles :

- Les concepts fondamentaux ont ete presentes et illustres
- Les exercices proposent une mise en pratique progressive
- Les resultats obtenus permettent de valider la comprehension

**Pour aller plus loin** : approfondir les aspects avances du sujet et explorer les liens avec d'autres domaines.